# Day 37: Agent Skills

Your agent carries all its knowledge upfront — every prompt, every instruction, always in context.

But what if it could load specialized knowledge **on-demand**? That's the Skills pattern — progressive disclosure for agents.

## Install

In [ ]:
%pip install langgraph langchain-google-genai langchain --quiet

## Setup

In [12]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
print("\u2705 Model ready")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Model ready


## Define Skills

Each skill is a specialized prompt — domain knowledge the agent loads only when needed.

In [13]:
SKILLS = {
    "write_sql": (
        "You are an SQL expert. Write clean, efficient SQL queries. "
        "Use CTEs for complex queries. Always add comments explaining the logic. "
        "Prefer explicit JOINs over implicit. Use meaningful aliases."
    ),
    "review_code": (
        "You are a code reviewer. Focus on: bugs, security issues, performance, readability. "
        "For each issue found, state: line/area, severity (high/medium/low), suggestion. "
        "Be concise. No praise — only actionable feedback."
    ),
    "explain_concept": (
        "You are a teacher. Explain concepts using: one simple analogy, "
        "one concrete example, and one sentence summary. "
        "Target audience: beginner developers. No jargon without explanation."
    ),
}

print(f"\u2705 {len(SKILLS)} skills defined: {', '.join(SKILLS.keys())}")

✅ 3 skills defined: write_sql, review_code, explain_concept


## The load_skill Tool

The agent calls this tool to load a skill's prompt into its context — progressive disclosure.

In [14]:
@tool
def load_skill(skill_name: str) -> str:
    """Load a specialized skill to enhance your capabilities.

    Available skills:
    - write_sql: Expert SQL query writing
    - review_code: Code review with actionable feedback
    - explain_concept: Beginner-friendly explanations

    Returns the skill's instructions. Follow them for the current task.
    """
    if skill_name in SKILLS:
        return f"SKILL LOADED: {SKILLS[skill_name]}"
    return f"Unknown skill: {skill_name}. Available: {', '.join(SKILLS.keys())}"

print("\u2705 load_skill tool defined")

✅ load_skill tool defined


## Build the Agent

Same graph pattern. The agent's system prompt tells it to load a skill before answering specialized questions.

In [15]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import ToolMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

tools = [load_skill]

class State(TypedDict):
    messages: Annotated[list, add_messages]

model_with_tools = model.bind_tools(tools)

SYSTEM = (
    "You are a helpful assistant with access to specialized skills. "
    "Before answering a specialized question, ALWAYS call load_skill first "
    "to get the right expertise. Then follow those instructions exactly."
)

In [16]:
def llm_node(state: State):
    messages = [SystemMessage(content=SYSTEM)] + state["messages"]
    return {"messages": [model_with_tools.invoke(messages)]}

def tool_node(state: State):
    results = []
    for call in state["messages"][-1].tool_calls:
        fn = {t.name: t for t in tools}[call["name"]]
        output = fn.invoke(call["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return {"messages": results}

def should_continue(state: State) -> Literal["tool_node", "__end__"]:
    return "tool_node" if state["messages"][-1].tool_calls else END

builder = StateGraph(State)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", should_continue, ["tool_node", END])
builder.add_edge("tool_node", "llm_node")
agent = builder.compile()
print("\u2705 Skills agent compiled")

✅ Skills agent compiled


## Test: SQL Skill

In [17]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Write a SQL query to find the top 5 customers by total order value, including their email."}]}
)

final = result["messages"][-1]
content = final.content if isinstance(final.content, str) else final.content[0]["text"]
print(content)

```sql
-- Select customer email and the sum of their order values
SELECT
  c.email,
  SUM(o.order_value) AS total_order_value
FROM
  customers AS c -- Alias the customers table as 'c'
JOIN
  orders AS o -- Alias the orders table as 'o'
ON
  c.customer_id = o.customer_id -- Join customers and orders tables on customer_id
GROUP BY
  c.customer_id,
  c.email -- Group by customer_id and email to aggregate order values for each customer
ORDER BY
  total_order_value DESC -- Order the results by total_order_value in descending order
LIMIT 5; -- Limit the output to the top 5 customers
```


## Test: Code Review Skill

In [18]:
code_to_review = """
def get_user(id):
    query = f"SELECT * FROM users WHERE id = {id}"
    result = db.execute(query)
    return result[0] if result else None
"""

result = agent.invoke(
    {"messages": [{"role": "user", "content": f"Review this code:{code_to_review}"}]}
)

final = result["messages"][-1]
content = final.content if isinstance(final.content, str) else final.content[0]["text"]
print(content)

Line/Area: `query = f"SELECT * FROM users WHERE id = {id}"`
Severity: High
Suggestion: This code is vulnerable to SQL injection. Use parameterized queries provided by your database library (e.g., `db.execute("SELECT * FROM users WHERE id = ?", (id,))`) instead of f-strings to prevent malicious input from altering query logic.

Line/Area: `db.execute(query)`
Severity: Medium
Suggestion: Consider using an Object-Relational Mapper (ORM) or a more explicit database client method for fetching a single user, which can improve readability and maintainability, and often provides built-in protection against SQL injection.


## Test: Explain Skill

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain what a hash table is."}]}
)

final = result["messages"][-1]
content = final.content if isinstance(final.content, str) else final.content[0]["text"]
print(content)

## Why Skills?

Without skills: stuff everything into the system prompt. Context window fills up. Agent gets confused.

With skills: agent carries a light system prompt and loads **only what it needs** for the current task. That's progressive disclosure — the same idea behind `llms.txt` and Claude's slash commands.

## Key Takeaways

1. **Skills = specialized prompts loaded on-demand** via a `load_skill` tool — not hardcoded in the system prompt
2. **Progressive disclosure** — the agent only loads knowledge when it needs it, keeping context lean
3. **Add new skills without changing the agent** — just add an entry to the SKILLS dict, the agent discovers it through the tool